# Random Forest & Gradient Boosting Prediction
Random Forest, XGBoost, and LightGBM tree-based models.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings, glob
warnings.filterwarnings('ignore')

# Global constants
EXTREME_T   = 28        # Extreme heat threshold (°C)
MAN_LAT     = 53.48
MAN_LON_360 = 360 - 2.24  # Manchester longitude in 0-360 format


In [ ]:
# ─── Global plot settings (journal style) ───────────────────────────────────
matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  9,
    'figure.dpi':       130,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.22,
    'grid.linestyle':   '--',
    'axes.axisbelow':   True,
})

C = dict(
    RF='#2166AC', XGB='#4DAC26', LGB='#E08214',
    LSTM='#762A83', CNN='#D6604D', CNN_LSTM='#01665E',
    Ensemble='#543005', obs='#252525',
    hist='#4393C3', fut='#D6604D', extreme='#B2182B',
)

def savefig(name):
    plt.savefig(f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')


## Prerequisites
Run `1_data_processing.ipynb` first.

In [152]:
# =============================================================================
# PART 5  树基集成模型
# =============================================================================
print('\n' + '=' * 70)
print('PART 5 — Tree-based Ensemble Models')
print('=' * 70)
 
# ── Random Forest ─────────────────────────────────────────────────────────────
print('\n[1/3] Random Forest ...')
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=12,
    min_samples_leaf=3, max_features='sqrt',
    random_state=42, n_jobs=-1)
rf_model.fit(X_train_s, y_train_s)
 
y_pred_rf_C  = inv_y(rf_model.predict(X_test_s))
rf_metrics   = evaluate_C(y_test_C, y_pred_rf_C, 'Random Forest')
rf_ext       = extreme_scores(y_test_C, y_pred_rf_C)
 
# ── XGBoost ───────────────────────────────────────────────────────────────────
print('\n[2/3] XGBoost ...')
xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, gamma=0.1,
    random_state=42, verbosity=0)
xgb_model.fit(X_train_s, y_train_s,
              eval_set=[(X_val_s, y_val_s)],
               verbose=False)
 
y_pred_xgb_C = inv_y(xgb_model.predict(X_test_s))
xgb_metrics  = evaluate_C(y_test_C, y_pred_xgb_C, 'XGBoost')
xgb_ext      = extreme_scores(y_test_C, y_pred_xgb_C)
 
# ── LightGBM ──────────────────────────────────────────────────────────────────
print('\n[3/3] LightGBM ...')
lgb_model = lgb.LGBMRegressor(
    n_estimators=400, num_leaves=63, max_depth=8,
    learning_rate=0.04, min_child_samples=10,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1)
lgb_model.fit(X_train_s, y_train_s,
              eval_set=[(X_val_s, y_val_s)],
              callbacks=[lgb.early_stopping(40, verbose=False)])
 
y_pred_lgb_C = inv_y(lgb_model.predict(X_test_s))
lgb_metrics  = evaluate_C(y_test_C, y_pred_lgb_C, 'LightGBM')
lgb_ext      = extreme_scores(y_test_C, y_pred_lgb_C)
 
tree_metrics_df = pd.DataFrame({
    'Random Forest': rf_metrics, 'XGBoost': xgb_metrics, 'LightGBM': lgb_metrics}).T
print('\nTree model summary (°C):')
print(tree_metrics_df.to_string())


PART 5 — Tree-based Ensemble Models

[1/3] Random Forest ...
  Random Forest      MAE=0.814°C  RMSE=1.095°C  R²=0.9607  MAPE=5.10%

[2/3] XGBoost ...
  XGBoost            MAE=0.531°C  RMSE=0.729°C  R²=0.9826  MAPE=3.42%

[3/3] LightGBM ...
  LightGBM           MAE=0.508°C  RMSE=0.700°C  R²=0.9839  MAPE=3.27%

Tree model summary (°C):
                    MAE      RMSE        R2      MAPE
Random Forest  0.814184  1.095259  0.960687  5.095938
XGBoost        0.530581  0.728823  0.982592  3.416182
LightGBM       0.507750  0.700144  0.983935  3.267940
